# Solution 02: Combined Multi-tab App

In [ ]:
import json, os
import gradio as gr
from gliner import GLiNER
from gliclass import GLiClassModel, ZeroShotClassificationPipeline
from transformers import AutoTokenizer

fixtures = os.path.normpath(os.path.join(os.getcwd(), "..", "fixtures", "input"))
with open(os.path.join(fixtures, "sample_texts.json")) as f:
    SAMPLE_TEXTS = json.load(f)

ENTITY_TYPES = ["malware", "vulnerability", "attack_technique", "affected_software", "threat_actor"]
ENTITY_COLORS = {
    "malware": "#ff6b6b",
    "vulnerability": "#ffa94d",
    "attack_technique": "#74c0fc",
    "affected_software": "#69db7c",
    "threat_actor": "#da77f2",
}
ATTACK_LABELS = ["ransomware", "phishing", "apt_attack", "ddos", "data_breach", "supply_chain_attack", "zero_day"]

print(f"✓ Loaded {len(SAMPLE_TEXTS)} sample texts")

In [ ]:
ner_model = GLiNER.from_pretrained("knowledgator/gliner-bi-edge-v2.0")
print("✓ GLiNER loaded")

cls_model = GLiClassModel.from_pretrained("knowledgator/gliclass-edge-v3.0")
cls_tokenizer = AutoTokenizer.from_pretrained("knowledgator/gliclass-edge-v3.0", add_prefix_space=True)
cls_pipeline = ZeroShotClassificationPipeline(
    cls_model, cls_tokenizer, classification_type='multi-label', device='cpu'
)
print("✓ GLiClass loaded")

In [ ]:
def run_ner(text: str, entity_types: list, threshold: float) -> list:
    if not text.strip() or not entity_types:
        return [(text, None)]
    entities = ner_model.predict_entities(text, entity_types, threshold=threshold)
    entities = sorted(entities, key=lambda e: e["start"])
    result = []
    cursor = 0
    for ent in entities:
        start, end, label = ent["start"], ent["end"], ent["label"]
        if start > cursor:
            result.append((text[cursor:start], None))
        result.append((text[start:end], label))
        cursor = end
    if cursor < len(text):
        result.append((text[cursor:], None))
    return result if result else [(text, None)]

## Part 1: Implement `run_cls`

In [ ]:
def run_cls(text: str, labels_str: str) -> dict:
    labels = [l.strip() for l in labels_str.split(',') if l.strip()]
    if not labels or not text.strip():
        return {}
    results = cls_pipeline(text, labels, threshold=0.0)[0]
    return {r['label']: round(float(r['score']), 4) for r in results}


scores = run_cls(SAMPLE_TEXTS[0]['text'], ', '.join(ATTACK_LABELS))
assert isinstance(scores, dict)
assert len(scores) > 0
assert all(isinstance(v, float) for v in scores.values())
assert all(k in ATTACK_LABELS for k in scores.keys())
top_label = max(scores, key=scores.get)
print(f"✓ run_cls works: top label = '{top_label}' ({scores[top_label]:.3f})")
assert top_label in {"ransomware", "apt_attack", "zero_day"}

## Part 2: Build Multi-tab `gr.Blocks` App

In [ ]:
with gr.Blocks(title="Cybersecurity NLP Toolkit") as combined_app:
    gr.Markdown("# Cybersecurity NLP Toolkit\nPowered by GLiNER + GLiClass edge models.")

    with gr.Tabs():
        with gr.Tab("Named Entity Recognition"):
            with gr.Row():
                with gr.Column():
                    t1_text = gr.Textbox(label="Text", lines=5)
                    t1_types = gr.CheckboxGroup(
                        choices=ENTITY_TYPES,
                        value=ENTITY_TYPES,
                        label="Entity types"
                    )
                    t1_thresh = gr.Slider(0.1, 0.9, value=0.4, step=0.05, label="Threshold")
                    t1_out = gr.HighlightedText(
                        label="Entities",
                        color_map=ENTITY_COLORS,
                        show_legend=True
                    )
                    gr.Button("Extract", variant="primary").click(
                        run_ner, [t1_text, t1_types, t1_thresh], t1_out
                    )
            gr.Examples([[t['text']] for t in SAMPLE_TEXTS[:3]], inputs=t1_text)

        with gr.Tab("Attack Classification"):
            with gr.Row():
                with gr.Column():
                    t2_text = gr.Textbox(label="Report text", lines=5)
                    t2_labels = gr.Textbox(
                        label="Labels (comma-separated)",
                        value=", ".join(ATTACK_LABELS)
                    )
                    t2_out = gr.Label(label="Scores", num_top_classes=7)
                    gr.Button("Classify", variant="primary").click(
                        run_cls, [t2_text, t2_labels], t2_out
                    )
            gr.Examples([[t['text']] for t in SAMPLE_TEXTS[:3]], inputs=t2_text)

assert 'combined_app' in dir()
assert hasattr(combined_app, 'launch')
print("✓ combined_app defined")
combined_app.launch()

## Part 3: Add `gr.State` for Custom Label Memory

In [ ]:
def run_cls_stateful(text: str, labels_str: str, saved_labels: str):
    if not labels_str.strip():
        labels_str = saved_labels
    else:
        saved_labels = labels_str
    scores = run_cls(text, labels_str)
    return scores, saved_labels


with gr.Blocks(title="Cybersecurity NLP Toolkit") as combined_app_stateful:
    gr.Markdown("# Cybersecurity NLP Toolkit\nPowered by GLiNER + GLiClass edge models.")

    with gr.Tabs():
        with gr.Tab("Named Entity Recognition"):
            with gr.Row():
                with gr.Column():
                    t1_text = gr.Textbox(label="Text", lines=5)
                    t1_types = gr.CheckboxGroup(
                        choices=ENTITY_TYPES,
                        value=ENTITY_TYPES,
                        label="Entity types"
                    )
                    t1_thresh = gr.Slider(0.1, 0.9, value=0.4, step=0.05, label="Threshold")
                    t1_out = gr.HighlightedText(
                        label="Entities",
                        color_map=ENTITY_COLORS,
                        show_legend=True
                    )
                    gr.Button("Extract", variant="primary").click(
                        run_ner, [t1_text, t1_types, t1_thresh], t1_out
                    )
            gr.Examples([[t['text']] for t in SAMPLE_TEXTS[:3]], inputs=t1_text)

        with gr.Tab("Attack Classification"):
            labels_state = gr.State(value=", ".join(ATTACK_LABELS))
            with gr.Row():
                with gr.Column():
                    t2_text = gr.Textbox(label="Report text", lines=5)
                    t2_labels = gr.Textbox(
                        label="Labels (comma-separated; clear to restore previous)",
                        value=", ".join(ATTACK_LABELS)
                    )
                    t2_out = gr.Label(label="Scores", num_top_classes=7)
                    gr.Button("Classify", variant="primary").click(
                        run_cls_stateful,
                        inputs=[t2_text, t2_labels, labels_state],
                        outputs=[t2_out, labels_state]
                    )
            gr.Examples([[t['text']] for t in SAMPLE_TEXTS[:3]], inputs=t2_text)

assert 'combined_app_stateful' in dir()

# Validate stateful logic
default_labels = ', '.join(ATTACK_LABELS)
scores1, state1 = run_cls_stateful(SAMPLE_TEXTS[0]['text'], default_labels, "")
assert state1 == default_labels
scores2, state2 = run_cls_stateful(SAMPLE_TEXTS[1]['text'], "", state1)
assert state2 == default_labels
assert len(scores2) > 0
print("✓ Stateful logic works")
combined_app_stateful.launch()